In [1]:
import oqupy
import oqupy.operators as op
import numpy as np
import matplotlib.pyplot as plt
import time

import numpy as np
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.optimize import minimize,Bounds

import oqupy as oqupy
from oqupy.iTEBD_TEMPO_useoqupybath import iTEBD_TEMPO_oqupy
from oqupy.process_tensor import TTInvariantProcessTensor
from oqupy.tti_tempo import TTITempo
import numpy as np
import matplotlib.pyplot as plt

from oqupy.gate_gradient import compute_dynamical_map,compute_dynamical_map_and_grad
from oqupy.gradient import state_gradient

from oqupy.gate_gradient import gate_chain_rule


pt_parameters = {'tmax':50,
                 'steps':100, 
                 'dkmax':500,
                 'epsrel':10**(-6),
                 'alpha':0.1,
                 'omega_cutoff':1,                 
                 'temp':0.131}


omega_cutoff = pt_parameters['omega_cutoff']
alpha = pt_parameters['alpha']
temperature = pt_parameters['temp']
epsrel = pt_parameters['epsrel']
t_max = pt_parameters['tmax']
# dt = 1./omega_cutoff/np.sqrt(3)
num_steps = pt_parameters['steps']
dt = t_max/num_steps
dkmax = pt_parameters['dkmax']

initial_state = op.spin_dm('x+')
target_state = op.spin_dm('x-')
target_derivative = target_state.T

t_list = np.linspace(0,t_max,num_steps+1)

correlations = oqupy.PowerLawSD(alpha=alpha,
                                zeta=1,
                                cutoff=omega_cutoff,
                                cutoff_type='exponential',
                                temperature=temperature)

bath = oqupy.Bath(op.sigma("z")/2.0, correlations)
parameters=oqupy.TempoParameters(dt=dt,epsrel=epsrel,dkmax=dkmax)

pt=TTITempo(bath,start_time=0.0,parameters=parameters)
process_tensor_tebd = pt.get_process_tensor()

def discrete_hamiltonian(hx,hy,hz):
    return hx*op.sigma('x')+ hy*op.sigma('y') + hz*op.sigma('z')
system = oqupy.ParameterizedSystem2ls(discrete_hamiltonian)

# ----- defining target unitary HERE ----
target_unitary=np.identity(2)

building influence functional: 100%|██████████| 500/500 [00:00<00:00, 673.36it/s] 

rank  27 (27, 4, 27)


In [2]:
from scipy.linalg import svd

import numpy as np
import numdifftools as nd
import matplotlib.pyplot as plt
from numdifftools import Jacobian

# ---- Fidelity over pure states
# ---------------- Superoperator -> Choi ----------------
def superop_to_choi_numpy(map_matrix):
    """
    map_matrix: 4x4 superoperator acting on 2x2 density matrix (flattened)
    """
    d = 2
    choi_matrix = np.zeros((d*d, d*d), dtype=np.complex128)
    for i in range(d):
        for j in range(d):
            e_ij = np.zeros((d,d), dtype=map_matrix.dtype)
            e_ij[i,j] = 1.0
            phi_e_ij = map_matrix @ e_ij.flatten()  # shape (4,)
            phi_e_ij_mat = phi_e_ij.reshape((d,d))
            choi_matrix += np.kron(e_ij, phi_e_ij_mat)
    return choi_matrix

# ---------------- Choi -> Kraus ----------------
def choi_to_kraus_numpy(choi):
    vals, vecs = np.linalg.eigh(choi)
    kraus_ops = []
    for val, vec in zip(vals, vecs.T):
        if val > 1e-20:
            K = np.sqrt(val) * vec.reshape((2,2))
            kraus_ops.append(K)
    return kraus_ops

# ---------------- Rotation fidelity ----------------
def rotation_fidelity_numpy(unitary, kraus_ops):
    d = 2
    first_term = np.zeros((d,d), dtype=np.complex128)
    second_term = 0.0
    for kraus in kraus_ops:
        M = unitary.conj().T @ kraus
        first_term += M.conj().T @ M
        second_term += abs(np.trace(M))**2
    first_term = np.trace(first_term)
    fidelity = (first_term + second_term) / (d*(d+1))
    return fidelity.real

# ---------------- Fidelity wrapper ----------------
def fidelity_wrapper_numpy(map_matrix, target_unitary):
    choi = superop_to_choi_numpy(map_matrix)
    kraus = choi_to_kraus_numpy(choi)
    return rotation_fidelity_numpy(target_unitary, kraus)

def unitary_to_superop(U):
    
    return np.kron(U, np.conj(U))

def superop_trace_distance(M, U):
    """Trace distance between map (superop) and unitary channel."""
    S_U = unitary_to_superop(U)
    diff = M - S_U
    svals = svd(diff, compute_uv=False)
    return 0.5*np.sum(np.abs(svals))/2

In [3]:
def map_and_grad(system,process_tensor_tebd,parameters,dt,num_steps):
        map_list,grads_list=compute_dynamical_map_and_grad(system,
                [process_tensor_tebd],
                parameters,
                dt,
                start_time=0,
                num_steps=num_steps)
        
        num_parameters = parameters.shape[1]

        propagators=system.get_propagators(dt,parameters)
        prop_derivs=system.get_propagator_derivatives(dt,parameters)

        dyn_map_derivatives=gate_chain_rule(adjoint_tensor=grads_list,propagators=propagators,dprop_dparam=prop_derivs,num_steps=num_steps,num_parameters=num_parameters,progress_type='bar')

        return map_list,dyn_map_derivatives

In [4]:
from numdifftools import Jacobian
import numpy as np

def fidelity_derivative_function(target_unitary):
    def fid_wrapper(map_matrix):
        choi = superop_to_choi_numpy(map_matrix)
        kraus = choi_to_kraus_numpy(choi)
        return rotation_fidelity_numpy(target_unitary, kraus)

    jacfun_re = Jacobian(lambda x: fid_wrapper(x.reshape(4,4)).real)
    jacfun_im = Jacobian(lambda x: fid_wrapper(x.reshape(4,4)).imag)

    def jacfun(map_matrix):
        x_flat = map_matrix.flatten()
        jac = jacfun_re(x_flat) + 1.0j * jacfun_im(x_flat)
        # Return a list of 16 scalars (derivative w.r.t. each element)
        return [jac[0, i] for i in range(16)]

    return jacfun

def trace_derivative_function(target_unitary):
    def trace_wrapper(map_matrix):

        return superop_trace_distance(map_matrix,target_unitary)

    jacfun_re = Jacobian(lambda x: trace_wrapper(x.reshape(4,4)).real)
    jacfun_im = Jacobian(lambda x: trace_wrapper(x.reshape(4,4)).imag)

    def jacfun(map_matrix):
        x_flat = map_matrix.flatten()
        jac = jacfun_re(x_flat) + 1.0j * jacfun_im(x_flat)
        # Return a list of 16 scalars (derivative w.r.t. each element)
        return [jac[0, i] for i in range(16)]

    return jacfun

def trace_and_gradient_numpy(map_matrix, target_unitary):
    """
    Compute fidelity and gradient for a single 4x4 map.
    Returns:
        fid_val: scalar fidelity
        grad_val: list of 16 arrays, each 4x4, derivative w.r.t. map element
    """
    grad_fn = trace_derivative_function(target_unitary)
    grad_val = grad_fn(map_matrix)

    tr_val = superop_trace_distance(map_matrix, target_unitary)
    return tr_val, grad_val


def fidelity_and_gradient_numpy(map_matrix, target_unitary):
    """
    Compute fidelity and gradient for a single 4x4 map.
    Returns:
        fid_val: scalar fidelity
        grad_val: list of 16 arrays, each 4x4, derivative w.r.t. map element
    """
    grad_fn = fidelity_derivative_function(target_unitary)
    grad_val = grad_fn(map_matrix)

    fid_val = fidelity_wrapper_numpy(map_matrix, target_unitary)
    return fid_val, grad_val


In [5]:
iteration=0

history=[]

def projected_grad(x, grad, p_bounds): 

    proj_grad = grad.copy() 
    for i, (xi, gi, bnd) in enumerate(zip(x, grad, p_bounds)): 
        lb, ub = bnd 
        if lb is not None and np.isclose(xi, lb) and gi > 0: 
            proj_grad[i] = 0.0 
        elif ub is not None and np.isclose(xi, ub) and gi < 0: 
            proj_grad[i] = 0.0 
        
    return proj_grad

# next check, do infid and grad of state fidelity using map 
def infidandgrad_numpy(paras,num_parameters,bounds=[(None,None)]):
    """""
    Take a numpy array [hx0, hz0, hx1, hz1, ...] over full timesteps and
    return the fidelity and gradient of the fidelity w.r.t. [hx0, hz0, hx1, hz1, ...]
    """

    # Reshape flat parameter list to form accepted by state_gradient: [[hx0,hz0],[hx1,hz1,]...]
    reshapedparas = [i for i in (paras.reshape((-1,num_parameters))).tolist() for j in range(3)]
    reshapedparas = np.array(reshapedparas)

    # Map + derivative of map w.r.t. parameters at each time step
    maps,map_grads=map_and_grad(system,process_tensor_tebd,reshapedparas,dt,num_steps)

    # Map gradients are computed over half time steps (add together adjacent grads)
    for i in range(0,map_grads.shape[0],2): 
        map_grads[i,:]=map_grads[i,:]+ map_grads[i+1,:]
            
    map_grads=map_grads[0::2]
    
    target_unitary=np.identity(2)

    # Calculation of final map fidelity and derivative of fidelity w.r.t. final map
    # (Here can swap out trace or fidelity unitary)
    fidelity,fidelity_grad=fidelity_and_gradient_numpy(maps[-1].T,target_unitary)
    #fidelity,fidelity_grad=trace_and_gradient_numpy(maps[-1],target_unitary)
    fidelity_grad= np.array(fidelity_grad).reshape(4,4)  # reshape back to 4x4

    # Sum over matrix indices (last two axes) to get derivative w.r.t each parameter
    # Result: (num_steps, num_parameters)
    grads_per_param = np.einsum("tpij,ij->tp", map_grads, fidelity_grad)

    # Flatten to match optimizer input: [hx0, hz0, hx1, hz1, ...]
    grads_flat = grads_per_param.flatten()

    proj_grad=projected_grad(paras,grads_flat,bounds)

    proj_norm=np.linalg.norm(proj_grad,ord=np.inf)

    global iteration
    iteration+=1
    global history
    history.append((iteration,1-fidelity.real,np.linalg.norm(grads_flat.real),proj_norm))
    print("[iteration {0}], infidelity: {1}, gradient norm: {2}".format(iteration,1-fidelity.real,np.linalg.norm(grads_flat.real)))

    # Return the minus the gradient as infidelity is being minimized 
    return 1-fidelity.real,-1.0*grads_flat.real

In [6]:
num_params=3
def state_infid_grad_test(paras):
    """""
    Take a numpy array [hx0, hz0, hx1, hz1, ...] over full timesteps and
    return the fidelity and gradient of the fidelity to the global target_derivative
    """
    # Reshape flat parameter list to form accepted by state_gradient: [[hx0,hz0],[hx1,hz1,]...]
    reshapedparas = [i for i in (paras.reshape((-1,num_params))).tolist() for j in range(3)]
    reshapedparas = np.array(reshapedparas)

    maps,map_grads=map_and_grad(system,process_tensor_tebd,reshapedparas,dt,num_steps)
    
    vec_Rho=initial_state.reshape(2**2) # row major
    fs_vec=maps[-1].T@vec_Rho # transpose here yields correct fid
    fs=fs_vec.reshape((2,2))

    fidelity=np.sum(fs*target_derivative.T)

    td_vec=target_derivative.reshape(2**2)
    gps=np.einsum("tpij,i,j->tp",map_grads,vec_Rho,td_vec) # no transpose here but one above... need to check consistency of gate with gradient ordering

    # Adding adjacent elements
    for i in range(0,gps.shape[0],2): 
        gps[i,:]=gps[i,:]+gps[i+1,:]
        
    gps=gps[0::2]


    # Return the minus the gradient as infidelity is being minimized 
    return 1-fidelity.real,(-1.0*gps.reshape((-1)).real).tolist()

def state_infid_grad(paras):
    """""
    Take a numpy array [hx0, hz0, hx1, hz1, ...] over full timesteps and
    return the fidelity and gradient of the fidelity to the global target_derivative
    """
    # Reshape flat parameter list to form accepted by state_gradient: [[hx0,hz0],[hx1,hz1,]...]
    reshapedparas = [i for i in (paras.reshape((-1,num_params))).tolist() for j in range(3)]
    reshapedparas = np.array(reshapedparas)

    gradient_dict = oqupy.state_gradient(
        system=system,
        initial_state=initial_state,
        target_derivative=target_derivative,
        process_tensors=[process_tensor_tebd],
        parameters=reshapedparas,
        num_steps=num_steps,
        progress_type='bar')
    
    
    fs=gradient_dict['final_state']
    gps=gradient_dict['gradient']
    fidelity=np.sum(fs*target_derivative.T)

    # Adding adjacent elements
    for i in range(0,gps.shape[0],2): 
        gps[i,:]=gps[i,:]+gps[i+1,:]
        
    gps=gps[0::2]

    # Return the minus the gradient as infidelity is being minimized 
    return 1-fidelity.real,(-1.0*gps.reshape((-1)).real).tolist()


In [7]:
# warning : takes ~ 250-300 minutes


z0 = np.zeros(num_steps)
x0 = np.ones(num_steps)*omega_cutoff
y0 = np.zeros(num_steps)

parameter_list=[item for pair in zip(x0, y0, z0) for item in pair]
# Flatten list for input to optimizer
x_bound = [-np.pi/2,np.pi/2]
y_bound = [0.0,0.0]
z_bound = [-np.pi/2,np.pi/2]

num_parameters=3
bounds = np.zeros((num_steps*num_parameters,2))

for i in range(0, num_parameters*num_steps,num_parameters):
        bounds[i] = x_bound
        #bounds[i+1] = y_bound
        bounds[i+1] = z_bound

fun = lambda x: infidandgrad_numpy(x, num_parameters = num_parameters, bounds = bounds)
optimization_result_test = minimize(
                        fun=fun,
                        x0=parameter_list,
                        method='L-BFGS-B',
                        jac=True,
                        #bounds=bounds,
                        options = {'disp':True, 'maxiter': 3}
)



/tmp/ipykernel_1302/872084286.py:23: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  optimization_result_test = minimize(


--> Apply chain rule:
 76.0%   76 of  100 [##############################----------] 00:00:03

KeyboardInterrupt: 

 76.0%   76 of  100 [##############################----------] 00:00:04

 76.0%   76 of  100 [##############################----------] 00:00:15

In [ ]:
reshapedparas=np.array([i for i in (np.array(optimization_result_test.x).reshape((-1,num_parameters))).tolist()])
plt.plot(reshapedparas[:,0])
plt.plot(reshapedparas[:,1])
plt.show()

In [ ]:
fig, axs = plt.subplots(nrows=4, ncols=1,figsize=(8,10)) 
fig.suptitle("Optimisation results")
fig.subplots_adjust(hspace=0.6)  # more vertical space between rows

field_labels = ["x","y","z"]
for i in range(0,num_parameters):
        axs[0].plot(t[:-1],optimization_result_test['x'][i::num_parameters],label=field_labels[i])
        axs[0].set_ylabel(r"$h_i$",rotation=0,fontsize=16)
        axs[0].set_xlabel("t")
        axs[0].legend()

basis_states=[op.SPIN_DM["x+"],op.SPIN_DM["y+"],op.SPIN_DM["z+"]] 
basis_labels=["|x+><x+|","|y+><y+|","|z+><z+|"]

for i,state in enumerate(basis_states):
        j=i+1
        # Input optimized controls into state_gradient to show dynamics of system under optimized fields
        optimized_dynamics = state_gradient(
                system=system,
                initial_state=np.array(state),
                target_derivative=target_derivative,
                process_tensors=[process_tensor_tebd],
                parameters=reshapedparas,
                num_steps=num_steps)

        dynamics = optimized_dynamics['dynamics']

        t, bloch_x =optimized_dynamics['dynamics'].expectations(op.sigma("x"))
        t, bloch_y = optimized_dynamics['dynamics'].expectations(op.sigma("y"))
        t, bloch_z = optimized_dynamics['dynamics'].expectations(op.sigma("z"))

        axs[j].plot(t,bloch_x,label='x')
        axs[j].plot(t,bloch_y,label='y')
        axs[j].plot(t,bloch_z,label='z')
        bloch_length = np.sqrt(bloch_x**2 +bloch_y**2 + bloch_z**2)
        axs[j].legend()
        axs[j].plot(t,bloch_length,label=r'$|\mathbf{\sigma}|$')
        axs[j].set_title(r"$\rho(0)={0}$".format(basis_labels[i]))
        axs[j].set_ylabel(r"$\langle \sigma \rangle$",rotation=0,fontsize=16)
        axs[j].set_xlabel("t")
